# Model v2: training on Google Colab's GPU

Use this when training on the Mac is too slow (RoBERTa-base). It runs the same training code as `sweet-truth model train`.

1. In Colab: **Runtime → Change runtime type → T4 GPU**.
2. Run the cells in order. The last one downloads a zip of the trained models.
3. On the Mac, unzip it in the repository root so the models land in `models/v2/`, then run `uv run sweet-truth model evaluate`.

In [ ]:
# Get the code (branch with the training package) and install it with uv.
!git clone --quiet --branch model-v2 https://github.com/bhawakshi-punia/sweet_truth.git
%cd sweet_truth
!pip install --quiet uv
!uv pip install --system --quiet -e ".[train]"

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch the runtime to a GPU")

In [ ]:
from sweet_truth.training.corpus import prepare
from sweet_truth.training.train import train_folds, train_model

TRANSFORMER = "roberta-base"  # or "electra-small"

prepare()
best = train_model(TRANSFORMER, gpu_id=0)
folds = train_folds(TRANSFORMER, gpu_id=0)
best, folds

In [ ]:
# Keep only the best version of each model, zip and download.
import shutil
from pathlib import Path

root = Path("models/v2") / TRANSFORMER
for last in root.rglob("model-last"):
    shutil.rmtree(last)
archive = shutil.make_archive(f"model_v2_{TRANSFORMER}", "zip", root_dir=".", base_dir=str(root))

from google.colab import files
files.download(archive)